# E-Commerce Marketplace Analytics — SQL Analysis

## Introduction

This notebook executes the SQL analysis against
`database/ecommerce.db` (built in `02_sqlite_database_setup.ipynb`) and
captures the results for review. It runs the four SQL files produced for
this stage of the project:

- `sql/01_business_queries.sql` — 20 business queries (6 basic, 8
  intermediate, 6 advanced junior)
- `sql/02_window_functions.sql` — 5 dedicated window function
  demonstrations
- `sql/03_cte_queries.sql` — 5 reusable CTE building blocks
- `sql/04_views.sql` — 4 persistent views for reuse in later notebooks

Every query answers a specific business question. This notebook does not
perform EDA, calculate KPIs in Python, build charts, or create dashboards —
it purely executes and displays SQL results. `04_python_eda.ipynb` picks up
from here in Python.


## Setup: Connect to the Database and Load Query Files

A small helper parses each `.sql` file into individual runnable
statements (respecting full-line comments), so every query can be
executed and displayed one at a time with its own label.


In [1]:
from __future__ import annotations

import re
import sqlite3
import warnings
from pathlib import Path
from typing import Dict, List, Tuple

import pandas as pd

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

PROJECT_ROOT = Path("..").resolve()
DB_PATH = PROJECT_ROOT / "database" / "ecommerce.db"
SQL_DIR = PROJECT_ROOT / "sql"
REPORTS_DIR = PROJECT_ROOT / "reports"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if not DB_PATH.exists():
    raise FileNotFoundError(
        f"Database not found at {DB_PATH}.\n"
        "Run notebooks/02_sqlite_database_setup.ipynb first to build it."
    )

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
print(f"Connected to {DB_PATH}")


Connected to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/database/ecommerce.db


In [2]:
def parse_sql_statements(sql_path: Path) -> List[str]:
    """Split a .sql file into individual executable statements.

    Full-line comments (lines starting with `--`) and blank lines are
    stripped before splitting on semicolons, so semicolons that appear
    inside comment text never cause an incorrect split.
    """
    text = sql_path.read_text(encoding="utf-8")
    kept_lines = [
        line for line in text.splitlines()
        if line.strip() and not line.strip().startswith("--")
    ]
    cleaned = "\n".join(kept_lines)
    statements = [s.strip() for s in cleaned.split(";") if s.strip()]
    # Drop bare PRAGMA statements from the display loop; they are executed
    # once at connection time above and do not return a result set.
    return [s for s in statements if not s.upper().startswith("PRAGMA")]


def run_and_display(connection: sqlite3.Connection, statement: str, label: str) -> pd.DataFrame:
    """Execute one SQL statement and return its result as a DataFrame."""
    df = pd.read_sql_query(statement, connection)
    print(f"--- {label} ---")
    print(df.to_string(index=False) if not df.empty else "(no rows returned)")
    print()
    return df


## Business Questions

The 20 business queries below cover Executive Overview, Sales, Customers,
Products, Categories, Regions, Delivery, Operations, Reviews, and
Marketplace Performance. Two seller-specific questions are intentionally
not implemented, since the `sellers` table was excluded from the core
schema — this is noted at the end of `sql/01_business_queries.sql` and
repeated in the analysis report.


## Basic Queries (B1–B6)

Executed directly from `sql/01_business_queries.sql`.


In [3]:
business_statements = parse_sql_statements(SQL_DIR / "01_business_queries.sql")
print(f"Parsed {len(business_statements)} business query statements.")

basic_labels = [
    "B1: Total revenue and total orders",
    "B2: Average order value",
    "B3: Unique customers",
    "B4: Top 10 products by revenue",
    "B5: Order status distribution",
    "B6: Average review score",
]
basic_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(basic_labels, business_statements[0:6]):
    basic_results[label] = run_and_display(conn, stmt, label)


Parsed 20 business query statements.
--- B1: Total revenue and total orders ---
 total_revenue  total_orders
    13591643.7         98666

--- B2: Average order value ---
 average_order_value
              137.75

--- B3: Unique customers ---
 unique_customers
            96096

--- B4: Top 10 products by revenue ---
                      product_id  total_revenue
bb50f2e236e5eea0100680137654686c       63885.00
6cdd53843498f92890544667809f1595       54730.20
d6160fb7873f184099d9bc95e30376af       48899.34
d1c427060a0f73f6b889a5c7c61f2ac4       47214.51
99a4788cb24856965c36a24e339b6058       43025.56
3dd2a17168ec895c781a9191c1e95ad7       41082.60
25c38557cf793876c5abdd5931f922db       38907.32
5f504b3a1c75b73d6151be81eb05bdc9       37733.90
53b36df67ebb7c41585e8d54d6772e08       37683.42
aca2eb7d00ea1a7b8ebd4e68314663af       37608.90

--- B5: Order status distribution ---
order_status  order_count  pct_of_orders
   delivered        96478          97.02
     shipped         1107       

## Intermediate Queries (I1–I8)


In [4]:
intermediate_labels = [
    "I1: Revenue by category",
    "I2: Revenue by state",
    "I3: Monthly revenue and order trend",
    "I4: Repeat customer rate",
    "I5: Average delivery time",
    "I6: Late delivery rate",
    "I7: Orders by weekday",
    "I8: Review score by delivery outcome",
]
intermediate_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(intermediate_labels, business_statements[6:14]):
    intermediate_results[label] = run_and_display(conn, stmt, label)


--- I1: Revenue by category ---
                                     category  total_revenue
                                health_beauty     1258681.34
                                watches_gifts     1205005.68
                               bed_bath_table     1036988.68
                               sports_leisure      988048.97
                        computers_accessories      911954.32
                              furniture_decor      729762.49
                                   cool_stuff      635290.85
                                   housewares      632248.66
                                         auto      592720.11
                                 garden_tools      485256.46
                                         toys      483946.60
                                         baby      411764.89
                                    perfumery      399124.87
                                    telephony      323667.53
                             office_furniture      27

## Advanced Junior Queries (A1–A6)


In [5]:
advanced_labels = [
    "A1: Top product per category (ROW_NUMBER)",
    "A2: State revenue ranking (RANK)",
    "A3: Running monthly revenue total (SUM OVER)",
    "A4: Month-over-month revenue growth (LAG)",
    "A5: Above-average-revenue customers",
    "A6: Top category per state (DENSE_RANK)",
]
advanced_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(advanced_labels, business_statements[14:20]):
    advanced_results[label] = run_and_display(conn, stmt, label)


--- A1: Top product per category (ROW_NUMBER) ---
                                     category                      top_product  product_revenue
                                health_beauty bb50f2e236e5eea0100680137654686c         63885.00
                                    computers d6160fb7873f184099d9bc95e30376af         48899.34
                        computers_accessories d1c427060a0f73f6b889a5c7c61f2ac4         47214.51
                               bed_bath_table 99a4788cb24856965c36a24e339b6058         43025.56
                                         baby 25c38557cf793876c5abdd5931f922db         38907.32
                                   cool_stuff 5f504b3a1c75b73d6151be81eb05bdc9         37733.90
                                watches_gifts 53b36df67ebb7c41585e8d54d6772e08         37683.42
                              furniture_decor aca2eb7d00ea1a7b8ebd4e68314663af         37608.90
                                 garden_tools 422879e10f46682990de24d770e7f83d        

## Window Functions

Executed directly from `sql/02_window_functions.sql` — five standalone,
technique-focused demonstrations (`ROW_NUMBER`, `RANK`, `DENSE_RANK`,
`SUM() OVER()`, `LAG`).


In [6]:
window_statements = parse_sql_statements(SQL_DIR / "02_window_functions.sql")
window_labels = [
    "ROW_NUMBER: most recent review per order",
    "RANK: category revenue ranking with ties",
    "DENSE_RANK: state satisfaction ranking with ties",
    "SUM() OVER(): cumulative monthly revenue",
    "LAG: month-over-month delivered order volume change",
]
window_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(window_labels, window_statements):
    window_results[label] = run_and_display(conn, stmt, label)


--- ROW_NUMBER: most recent review per order ---


IOPub data rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_data_rate_limit`.

Current values:
ServerApp.iopub_data_rate_limit=1000000.0 (bytes/sec)
ServerApp.rate_limit_window=3.0 (secs)



--- DENSE_RANK: state satisfaction ranking with ties ---
customer_state  average_review_score  satisfaction_rank
            AM                  4.20                  1
            AP                  4.19                  2
            PR                  4.18                  3
            SP                  4.18                  4
            MG                  4.14                  5
            RS                  4.14                  6
            TO                  4.10                  7
            MS                  4.10                  8
            MT                  4.10                  9
            RN                  4.10                 10
            SC                  4.08                 11
            AC                  4.08                 12
            DF                  4.07                 13
            RO                  4.06                 14
            GO                  4.05                 15
            ES                  4.04           

## Common Table Expressions (CTEs)

Executed directly from `sql/03_cte_queries.sql` — five reusable CTE
building blocks: clean order-level revenue, customer revenue summary,
monthly revenue table, delivery performance, and review summary.


In [7]:
cte_statements = parse_sql_statements(SQL_DIR / "03_cte_queries.sql")
cte_labels = [
    "Clean order-level revenue (top 10 by revenue)",
    "Customer revenue summary (top 10 by revenue)",
    "Monthly revenue table",
    "Delivery performance summary",
    "Review score summary by category",
]
cte_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(cte_labels, cte_statements):
    cte_results[label] = run_and_display(conn, stmt, label)


--- Clean order-level revenue (top 10 by revenue) ---
                        order_id               customer_unique_id order_status order_purchase_timestamp  has_date_anomaly  order_revenue
03caa2c082116e1d31e67e9ae3700499 0a0a92112bd4c708ca5fde585afaa872    delivered      2017-09-29 15:24:52                 0        13440.0
736e1922ae60d0d6a89247b851902527 763c8b1c9c68a0229c42c9fc6f662b93    delivered      2018-07-15 14:49:44                 0         7160.0
0812eb902a67711a1cb742b3cdaa65ae dc4802a71eae9be1dd28f5d788ceb526    delivered      2017-02-12 20:37:36                 0         6735.0
fefacc66af859508bf1a7934eab1e97f 459bef486812aa25204be022145caa62    delivered      2018-07-25 18:10:17                 0         6729.0
f5136e38d1a14a4dbd87dff67da82701 ff4159b92c40ebe40454e3e6a7c35ed6    delivered      2017-05-24 18:14:34                 0         6499.0
2cc9089445046817a7539d90805e6e5a 4007669dec559734d6f53e029e360987    delivered      2017-11-24 11:03:35                 0   

## Views

Executes `sql/04_views.sql` to (re)create the four persistent views, then
queries each one to confirm it works. These views remain in the database
for `04_python_eda.ipynb` to query directly.


In [8]:
views_statements = parse_sql_statements(SQL_DIR / "04_views.sql")
# The first 8 statements alternate DROP VIEW IF EXISTS / CREATE VIEW for
# each of the 4 views (no result set); the remaining 4 are confirmation
# SELECTs against each view.
for stmt in views_statements[:8]:
    conn.execute(stmt)
conn.commit()
print("Views created:", [row[0] for row in conn.execute(
    "SELECT name FROM sqlite_master WHERE type='view' ORDER BY name;"
).fetchall()])


Views created: ['vw_category_summary', 'vw_customer_summary', 'vw_delivery_performance', 'vw_monthly_sales']


In [9]:
view_labels = [
    "vw_customer_summary (top 5 by revenue)",
    "vw_delivery_performance (sample)",
    "vw_monthly_sales",
    "vw_category_summary",
]
view_results: Dict[str, pd.DataFrame] = {}
for label, stmt in zip(view_labels, views_statements[8:12]):
    view_results[label] = run_and_display(conn, stmt, label)


--- vw_customer_summary (top 5 by revenue) ---
              customer_unique_id customer_state  order_count  total_revenue  avg_order_value
0a0a92112bd4c708ca5fde585afaa872             RJ            1        13440.0          13440.0
da122df9eeddfedc1dc1f5349a1a690c             RJ            2         7388.0           3694.0
763c8b1c9c68a0229c42c9fc6f662b93             ES            1         7160.0           7160.0
dc4802a71eae9be1dd28f5d788ceb526             MS            1         6735.0           6735.0
459bef486812aa25204be022145caa62             ES            1         6729.0           6729.0

--- vw_delivery_performance (sample) ---
                        order_id               customer_unique_id order_purchase_timestamp order_delivered_customer_date order_estimated_delivery_date  delivery_days  is_late
e481f51cbdc54678b7cc49136f2d6af7 7c396fd4830fd04220f754e42b4e5bff      2017-10-02 10:56:33           2017-10-10 21:25:13                    2017-10-18       8.436574        0
53c

## Key SQL Learnings

Reflections worth being able to articulate in an interview:

- **`ROW_NUMBER()` vs `RANK()` vs `DENSE_RANK()`** behave identically when
  there are no ties, and differ only in how they handle them — choosing
  the right one depends on whether tied items should occupy distinct
  positions, share a position with a gap after, or share a position with
  no gap.
- **Window functions vs `GROUP BY`:** a `GROUP BY` collapses rows into one
  per group; a window function (`OVER (...)`) keeps every original row
  while adding a calculated column computed across a defined window —
  this is why running totals and row-level rankings need `OVER()` rather
  than plain aggregation.
- **CTEs vs subqueries:** a CTE is not a performance optimization in
  SQLite (it does not create a materialized temp table by default) — its
  value here is entirely readability: naming an intermediate result
  (`monthly_revenue_table`, `delivery_performance`) makes a multi-step
  calculation easy to follow and easy to reuse within the same query.
- **Views vs CTEs:** a CTE only exists for the duration of one query; a
  `VIEW` persists in the database and can be queried by name from any
  future script or notebook — which is exactly why Section "Views" turns
  the most frequently reused CTEs into permanent views for reuse in the
  Python EDA notebook.
- **`STRFTIME` for date truncation:** SQLite has no native `DATE_TRUNC`;
  `STRFTIME('%Y-%m', ...)` is the SQLite-idiomatic way to truncate a
  timestamp to a calendar month for grouping.
- **`JULIANDAY()` for date arithmetic:** SQLite stores dates as text by
  default, so `JULIANDAY(a) - JULIANDAY(b)` is the standard way to get a
  numeric day difference between two timestamp columns.


## Generating the SQL Analysis Report

Finally, `reports/sql_analysis_report.md` is generated directly from the
result sets captured above — every number in that report traces back to
an actual query executed in this notebook.


In [10]:
def build_sql_analysis_report(
    basic_results: Dict[str, pd.DataFrame],
    intermediate_results: Dict[str, pd.DataFrame],
    advanced_results: Dict[str, pd.DataFrame],
    window_results: Dict[str, pd.DataFrame],
    cte_results: Dict[str, pd.DataFrame],
    view_results: Dict[str, pd.DataFrame],
) -> str:
    """Assemble reports/sql_analysis_report.md content from this notebook's live query results."""
    lines: List[str] = []
    lines.append("# SQL Analysis Report — E-Commerce Marketplace Analytics\n")
    lines.append(
        "This report is generated programmatically from `03_sql_analysis.ipynb`'s own "
        "query results, run against `database/ecommerce.db`. Every figure below reflects "
        "the actual database at the time this notebook was executed.\n"
    )

    lines.append("## Overview\n")
    lines.append(
        "20 business queries (6 basic, 8 intermediate, 6 advanced junior), 5 dedicated "
        "window function demonstrations, 5 reusable CTEs, and 4 persistent views were "
        "executed against database/ecommerce.db.\n"
    )

    lines.append("## Business Questions Answered\n")
    all_sections = {
        "Basic": basic_results,
        "Intermediate": intermediate_results,
        "Advanced Junior": advanced_results,
    }
    for section_name, results in all_sections.items():
        lines.append(f"### {section_name}\n")
        for label, df in results.items():
            lines.append(f"**{label}**\n")
            if df.empty:
                lines.append("(no rows returned)\n")
            else:
                lines.append(df.head(10).to_markdown(index=False))
                lines.append("\n")

    lines.append("## SQL Techniques Used\n")
    lines.append(
        "- Basic: SELECT, WHERE, GROUP BY, HAVING, ORDER BY, LIMIT, COUNT, SUM, AVG\n"
        "- Intermediate: INNER JOIN, LEFT JOIN, CASE WHEN, date aggregation (STRFTIME), "
        "subqueries, multiple joins\n"
        "- Advanced: CTEs, ROW_NUMBER(), RANK(), DENSE_RANK(), LAG(), SUM() OVER(), "
        "running totals, month-over-month growth, top-N-per-group patterns\n"
        "- Views: 4 persistent views wrapping the most frequently reused aggregations\n"
    )

    lines.append("## Key Findings\n")
    lines.append(
        "Findings are summarized qualitatively here since specific figures are already "
        "shown in full above and will vary with the real dataset. Review the Basic and "
        "Intermediate sections above for exact revenue, order, delivery, and review-score "
        "figures, and the Advanced section for ranking and trend results.\n"
    )

    lines.append("## Database Coverage\n")
    lines.append(
        "All 6 core tables (`categories`, `customers`, `products`, `orders`, `order_items`, "
        "`order_reviews`) are used across this query set. The `sellers`, `order_payments`, "
        "and `geolocation` tables remain unused, consistent with the project's original scope decision.\n"
    )

    lines.append("## Limitations\n")
    lines.append(
        "- Two specification business questions (seller count and seller revenue "
        "concentration) are not implemented, since the `sellers` table was not loaded "
        "into the core schema.\n"
        "- No profit, margin, or discount analysis is possible (no such data exists in "
        "this dataset).\n"
        "- Delivery and review analyses exclude rows flagged with `has_date_anomaly = 1` "
        "during data cleaning, to avoid a small number of logically inconsistent "
        "timestamps distorting delivery-time calculations.\n"
    )

    lines.append("## Preparation for Python Analysis\n")
    lines.append(
        "The four views created here (`vw_customer_summary`, "
        "`vw_delivery_performance`, `vw_monthly_sales`, `vw_category_summary`) are "
        "designed to be queried directly from `04_python_eda.ipynb` via pandas' "
        "`read_sql_query`, avoiding the need to re-derive the same joins in Python.\n"
    )

    return "\n".join(lines)


report_markdown = build_sql_analysis_report(
    basic_results=basic_results,
    intermediate_results=intermediate_results,
    advanced_results=advanced_results,
    window_results=window_results,
    cte_results=cte_results,
    view_results=view_results,
)

report_path = REPORTS_DIR / "sql_analysis_report.md"
report_path.write_text(report_markdown, encoding="utf-8")
print(f"SQL analysis report written to {report_path}")

conn.close()
print("Connection closed.")


SQL analysis report written to /Users/alinaamangeldieva/Downloads/ecommerce-marketplace-analytics FINAL/reports/sql_analysis_report.md
Connection closed.


## Next Notebook

**`04_python_eda.ipynb`** will pick up directly from the four views created
here, querying `vw_customer_summary`, `vw_delivery_performance`,
`vw_monthly_sales`, and `vw_category_summary` via pandas to calculate every
KPI, engineer derived features (delivery days, purchase month/weekday,
revenue segments, etc.), and perform exploratory analysis in pandas/NumPy —
still without producing charts, which begin in that notebook.
